In [1]:
!pip install ipywidgets pandas scikit-learn

In [2]:
import random
import pandas as pd
import ipywidgets as widgets

from IPython.display import display, clear_output
from sklearn.tree import DecisionTreeClassifier

In [3]:
questions = {
    "Beginner": [
        {
            "question": "What does AI stand for?",
            "options": [
                "Artificial Intelligence",
                "Automated Internet",
                "Advanced Interface",
                "Artificial Interaction"
            ],
            "answer": "Artificial Intelligence"
        },
        {
            "question": "Which of the following is an example of an input device?",
            "options": ["Keyboard", "Monitor", "Speaker", "Projector"],
            "answer": "Keyboard"
        },
        {
            "question": "What does UI stand for?",
            "options": [
                "User Interface",
                "Universal Internet",
                "User Intelligence",
                "Unified Interaction"
            ],
            "answer": "User Interface"
        }
    ],

    "Intermediate": [
        {
            "question": "Which AI technique learns patterns from data?",
            "options": [
                "Machine Learning",
                "HTML",
                "CSS",
                "File Compression"
            ],
            "answer": "Machine Learning"
        },
        {
            "question": "What is the primary purpose of an adaptive user interface?",
            "options": [
                "Adjust the interface based on user behavior",
                "Increase file size",
                "Disable personalization",
                "Prevent user interaction"
            ],
            "answer": "Adjust the interface based on user behavior"
        },
        {
            "question": "Which data can help an adaptive system personalize an interface?",
            "options": [
                "User interaction history",
                "Computer color only",
                "File extension only",
                "Screen manufacturer"
            ],
            "answer": "User interaction history"
        }
    ],

    "Advanced": [
        {
            "question": "Which machine learning approach can predict a user's preferred interface?",
            "options": [
                "Classification",
                "File encryption",
                "HTML rendering",
                "Data compression"
            ],
            "answer": "Classification"
        },
        {
            "question": "What is a major ethical concern with AI-based adaptive interfaces?",
            "options": [
                "User privacy",
                "Keyboard size",
                "Monitor weight",
                "File naming"
            ],
            "answer": "User privacy"
        },
        {
            "question": "An Intelligent User Interface primarily combines HCI with:",
            "options": [
                "Artificial Intelligence",
                "Mechanical engineering",
                "File management",
                "Computer manufacturing"
            ],
            "answer": "Artificial Intelligence"
        }
    ]
}

In [4]:
# Training examples:
# [accuracy percentage, number of questions answered]

X_train = [
    [20, 2],
    [30, 3],
    [40, 4],
    [50, 4],
    [60, 5],
    [70, 5],
    [80, 5],
    [90, 6],
    [100, 6]
]

# 0 = Beginner
# 1 = Intermediate
# 2 = Advanced

y_train = [0, 0, 0, 1, 1, 1, 2, 2, 2]

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

difficulty_names = {
    0: "Beginner",
    1: "Intermediate",
    2: "Advanced"
}

print("Adaptive AI model trained successfully!")

Adaptive AI model trained successfully!


In [5]:
user_data = {
    "questions_answered": 0,
    "correct_answers": 0,
    "incorrect_answers": 0,
    "difficulty": "Beginner",
    "history": []
}

current_question = None

In [6]:
def calculate_accuracy():

    if user_data["questions_answered"] == 0:
        return 0

    return (
        user_data["correct_answers"]
        / user_data["questions_answered"]
    ) * 100


def adapt_difficulty():

    # Keep the first two questions at Beginner level
    # so that we have enough interaction data.
    if user_data["questions_answered"] < 2:
        return "Beginner"

    accuracy = calculate_accuracy()

    prediction = model.predict(
        [[accuracy, user_data["questions_answered"]]]
    )[0]

    return difficulty_names[prediction]

In [7]:
title = widgets.HTML(
    value="""
    <h2>AI-Based Adaptive Learning Interface</h2>
    <p>
    This interface adapts question difficulty according
    to your performance.
    </p>
    """
)

difficulty_display = widgets.HTML()

question_display = widgets.HTML()

answer_options = widgets.RadioButtons(
    options=[],
    description="Answer:"
)

submit_button = widgets.Button(
    description="Submit Answer",
    button_style="success"
)

next_button = widgets.Button(
    description="Next Question",
    button_style="info",
    disabled=True
)

feedback = widgets.HTML()

performance = widgets.HTML()

output = widgets.Output()

In [8]:
def load_question():

    global current_question

    feedback.value = ""

    difficulty = user_data["difficulty"]

    current_question = random.choice(
        questions[difficulty]
    )

    difficulty_display.value = (
        f"<h4>Current Difficulty: {difficulty}</h4>"
    )

    question_display.value = (
        f"<b>{current_question['question']}</b>"
    )

    answer_options.options = current_question["options"]
    answer_options.value = None

    submit_button.disabled = False
    next_button.disabled = True

In [9]:
def submit_answer(button):

    if answer_options.value is None:
        feedback.value = (
            "<p><b>Please select an answer.</b></p>"
        )
        return

    selected_answer = answer_options.value
    correct_answer = current_question["answer"]

    user_data["questions_answered"] += 1

    if selected_answer == correct_answer:

        user_data["correct_answers"] += 1

        feedback.value = """
        <h4>Correct!</h4>
        <p>
        Great work. The system will use your performance
        to determine the appropriate difficulty level.
        </p>
        """

        result = "Correct"

    else:

        user_data["incorrect_answers"] += 1

        feedback.value = f"""
        <h4>Incorrect</h4>
        <p>
        The correct answer is:
        <b>{correct_answer}</b>
        </p>
        <p>
        The adaptive system may reduce or maintain the
        difficulty to support your learning.
        </p>
        """

        result = "Incorrect"

    accuracy = calculate_accuracy()

    user_data["history"].append({
        "Question": current_question["question"],
        "Selected Answer": selected_answer,
        "Correct Answer": correct_answer,
        "Result": result,
        "Difficulty": user_data["difficulty"],
        "Accuracy": round(accuracy, 2)
    })

    new_difficulty = adapt_difficulty()

    user_data["difficulty"] = new_difficulty

    performance.value = f"""
    <hr>
    <b>Questions Answered:</b>
    {user_data['questions_answered']}<br>

    <b>Correct:</b>
    {user_data['correct_answers']}<br>

    <b>Incorrect:</b>
    {user_data['incorrect_answers']}<br>

    <b>Accuracy:</b>
    {accuracy:.1f}%<br>

    <b>Next Difficulty:</b>
    {new_difficulty}
    """

    submit_button.disabled = True
    next_button.disabled = False

In [10]:
def next_question(button):
    load_question()


submit_button.on_click(submit_answer)
next_button.on_click(next_question)

In [11]:
display(
    title,
    difficulty_display,
    question_display,
    answer_options,
    submit_button,
    next_button,
    feedback,
    performance
)

load_question()

HTML(value='\n    <h2>AI-Based Adaptive Learning Interface</h2>\n    <p>\n    This interface adapts question d…

HTML(value='')

HTML(value='')

RadioButtons(description='Answer:', options=(), value=None)

Button(button_style='success', description='Submit Answer', style=ButtonStyle())

Button(button_style='info', description='Next Question', disabled=True, style=ButtonStyle())

HTML(value='')

HTML(value='')

In [12]:
history_df = pd.DataFrame(user_data["history"])

history_df

,Question,Selected Answer,Correct Answer,Result,Difficulty,Accuracy
0,Which of the following is an example of an inp...,Keyboard,Keyboard,Correct,Beginner,100.0


In [13]:
history_df = pd.DataFrame(user_data["history"])

history_df.to_csv(
    "adaptive_ui_user_history.csv",
    index=False
)

print("User interaction history exported successfully.")

User interaction history exported successfully.
